# Materialization Pipeline Test
Tests the full step-by-step flow:
```
offline-store/delta/train
    ↓ Step 1: DuckDB reads Delta → PyArrow
    ↓ Step 2: Register as DuckDB in-memory table
    ↓ Step 3: SQL transforms (STAGING_SQL)
    ↓ Step 4: COPY TO S3 (httpfs)
offline-store/parquet/users/staged.parquet
```
**Prerequisites:** Docker stack running + `preprocess.py` executed.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import duckdb
import pyarrow as pa
from deltalake import DeltaTable

MINIO_ENDPOINT   = 'http://localhost:9000'
MINIO_ACCESS_KEY = 'minioadmin'
MINIO_SECRET_KEY = 'minioadmin123'

DELTA_PATH     = 's3://offline-store/delta/train'
PARQUET_OUTPUT = 's3://offline-store/parquet/users/staged.parquet'

STORAGE_OPTIONS = {
    'endpoint_url':           MINIO_ENDPOINT,
    'aws_access_key_id':      MINIO_ACCESS_KEY,
    'aws_secret_access_key':  MINIO_SECRET_KEY,
    'aws_allow_http':         'true',
    'aws_region':             'us-east-1',
}

STAGING_SQL = """
    SELECT
        user_id,
        event_timestamp,
        gender_idx,
        age_idx,
        occupation,
        target,
        target_time
    FROM train
    WHERE event_timestamp IS NOT NULL
"""

EXPECTED_COLUMNS = {'user_id', 'event_timestamp', 'gender_idx', 'age_idx', 'occupation', 'target', 'target_time'}

print('Setup complete.')

Setup complete.


## Step 1 — Delta Lake exists at s3://offline-store/delta/train

In [2]:
assert DeltaTable.is_deltatable(DELTA_PATH, storage_options=STORAGE_OPTIONS), \
    f'FAIL: No Delta table at {DELTA_PATH}. Run preprocess.py first.'

dt = DeltaTable(DELTA_PATH, storage_options=STORAGE_OPTIONS)
print(f'PASS: Delta table found at {DELTA_PATH}')
print(f'  Version  : {dt.version()}')
print(f'  Schema   : {[f.name for f in dt.schema().fields]}')

PASS: Delta table found at s3://offline-store/delta/train
  Version  : 2
  Schema   : ['user_id', 'event_timestamp', 'item_seq', 'genre_seq', 'time_seq', 'target', 'target_time', 'age_idx', 'gender_idx', 'occupation']


## Step 2 — Delta → PyArrow

In [3]:
arrow_table = dt.to_pyarrow_table()

assert isinstance(arrow_table, pa.Table),        'FAIL: to_pyarrow() did not return a pa.Table'
assert arrow_table.num_rows > 0,                 'FAIL: Delta table is empty'
assert 'user_id'         in arrow_table.schema.names, 'FAIL: missing user_id'
assert 'event_timestamp' in arrow_table.schema.names, 'FAIL: missing event_timestamp'

print(f'PASS: Delta → PyArrow  ({arrow_table.num_rows:,} rows, {arrow_table.num_columns} columns)')
print(f'  Columns: {arrow_table.schema.names}')

PASS: Delta → PyArrow  (770,166 rows, 10 columns)
  Columns: ['user_id', 'event_timestamp', 'item_seq', 'genre_seq', 'time_seq', 'target', 'target_time', 'age_idx', 'gender_idx', 'occupation']


## Step 3 — Register PyArrow table in DuckDB

In [4]:
conn = duckdb.connect(database=':memory:')
conn.register('train', arrow_table)

row_count = conn.execute('SELECT COUNT(*) FROM train').fetchone()[0]
col_names = [c[0] for c in conn.execute('DESCRIBE train').fetchall()]

assert row_count == arrow_table.num_rows, \
    f'FAIL: DuckDB row count {row_count} != Arrow row count {arrow_table.num_rows}'
assert 'user_id' in col_names,         'FAIL: user_id not in DuckDB table'
assert 'event_timestamp' in col_names, 'FAIL: event_timestamp not in DuckDB table'

print(f'PASS: Registered in DuckDB as "train"  ({row_count:,} rows)')
print(f'  Columns: {col_names}')

PASS: Registered in DuckDB as "train"  (770,166 rows)
  Columns: ['user_id', 'event_timestamp', 'item_seq', 'genre_seq', 'time_seq', 'target', 'target_time', 'age_idx', 'gender_idx', 'occupation']


## Step 4 — SQL transforms (STAGING_SQL)

In [5]:
df_staged = conn.execute(STAGING_SQL).df()

assert not df_staged.empty,                               'FAIL: STAGING_SQL returned 0 rows'
assert EXPECTED_COLUMNS == set(df_staged.columns),        f'FAIL: columns mismatch. Got {set(df_staged.columns)}'
assert df_staged['event_timestamp'].notna().all(),         'FAIL: event_timestamp has nulls'
assert df_staged['user_id'].notna().all(),                 'FAIL: user_id has nulls'
assert set(df_staged['gender_idx'].unique()) <= {0, 1},   'FAIL: gender_idx values outside {0,1}'
assert df_staged['age_idx'].min() >= 0,                   'FAIL: negative age_idx'
assert df_staged['target'].min() >= 1,                    'FAIL: target (movie_id) < 1'
assert set(df_staged['target_time'].unique()) <= {0,1,2}, 'FAIL: target_time outside {0,1,2}'

print(f'PASS: SQL transforms applied  ({len(df_staged):,} rows)')
print(f'  Columns       : {list(df_staged.columns)}')
print(f'  gender_idx    : {sorted(df_staged["gender_idx"].unique())}')
print(f'  target_time   : {sorted(df_staged["target_time"].unique())}')
display(df_staged.head(3))

PASS: SQL transforms applied  (770,166 rows)
  Columns       : ['user_id', 'event_timestamp', 'gender_idx', 'age_idx', 'occupation', 'target', 'target_time']
  gender_idx    : [np.int64(0), np.int64(1)]
  target_time   : [np.int64(0), np.int64(1), np.int64(2)]


,user_id,event_timestamp,gender_idx,age_idx,occupation,target,target_time
0,1,2001-01-01 05:00:55+07:00,0,0,10,935,2
1,1,2001-01-01 05:00:55+07:00,0,0,10,1152,2
2,1,2001-01-01 05:00:55+07:00,0,0,10,1540,2


## Step 5 — COPY TO S3 (httpfs)

In [6]:
endpoint = MINIO_ENDPOINT.replace('http://', '').replace('https://', '')

conn.execute('INSTALL httpfs; LOAD httpfs;')
conn.execute(f"""
    SET s3_endpoint          = '{endpoint}';
    SET s3_access_key_id     = '{MINIO_ACCESS_KEY}';
    SET s3_secret_access_key = '{MINIO_SECRET_KEY}';
    SET s3_use_ssl           = false;
    SET s3_url_style         = 'path';
""")

conn.execute(f"""
    COPY ({STAGING_SQL})
    TO '{PARQUET_OUTPUT}'
    (FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)
""")

conn.close()
print(f'PASS: COPY TO S3 complete → {PARQUET_OUTPUT}')

PASS: COPY TO S3 complete → s3://offline-store/parquet/users/staged.parquet


## Step 6 — Verify Parquet output in MinIO

In [7]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
)

resp = s3.list_objects_v2(Bucket='offline-store', Prefix='parquet/users/staged.parquet')
files = resp.get('Contents', [])

assert len(files) > 0, 'FAIL: staged.parquet not found in s3://offline-store/parquet/users/'

file_size_kb = files[0]['Size'] / 1024
print('PASS: staged.parquet exists in MinIO')
print(f'  Key  : {files[0]["Key"]}')
print(f'  Size : {file_size_kb:.1f} KB')

PASS: staged.parquet exists in MinIO
  Key  : parquet/users/staged.parquet
  Size : 4335.2 KB


## Step 7 — Read back Parquet and validate schema

In [8]:
import io

import pyarrow.parquet as pq

obj = s3.get_object(Bucket='offline-store', Key='parquet/users/staged.parquet')
df_out = pq.read_table(io.BytesIO(obj['Body'].read())).to_pandas()

assert not df_out.empty,                            'FAIL: staged.parquet is empty'
assert EXPECTED_COLUMNS == set(df_out.columns),     f'FAIL: schema mismatch. Got {set(df_out.columns)}'
assert len(df_out) == len(df_staged),               f'FAIL: row count mismatch ({len(df_out)} vs {len(df_staged)})'
assert df_out['user_id'].notna().all(),              'FAIL: user_id has nulls in Parquet'
assert df_out['event_timestamp'].notna().all(),      'FAIL: event_timestamp has nulls in Parquet'

print('PASS: Parquet validated')
print(f'  Rows    : {len(df_out):,}')
print(f'  Columns : {list(df_out.columns)}')
print('  Dtypes  :')
for col, dtype in df_out.dtypes.items():
    print(f'    {col:<16} {dtype}')
display(df_out.head(3))

PASS: Parquet validated
  Rows    : 770,166
  Columns : ['user_id', 'event_timestamp', 'gender_idx', 'age_idx', 'occupation', 'target', 'target_time']
  Dtypes  :
    user_id          int64
    event_timestamp  datetime64[us, UTC]
    gender_idx       int64
    age_idx          int64
    occupation       int64
    target           int64
    target_time      int64


,user_id,event_timestamp,gender_idx,age_idx,occupation,target,target_time
0,1,2000-12-31 22:00:55+00:00,0,0,10,935,2
1,1,2000-12-31 22:00:55+00:00,0,0,10,1152,2
2,1,2000-12-31 22:00:55+00:00,0,0,10,1540,2


## Full pipeline — run via `delta_to_duckdb_to_parquet()`

In [9]:
from src.features.materialization import delta_to_duckdb_to_parquet

row_count = delta_to_duckdb_to_parquet()

assert row_count > 0, 'FAIL: delta_to_duckdb_to_parquet() returned 0 rows'

print('PASS: Full pipeline complete')
print(f'  Rows staged : {row_count:,}')
print(f'  Delta path  : {"s3://offline-store/delta/train"}')
print(f'  Parquet out : {"s3://offline-store/parquet/users/staged.parquet"}')

PASS: Full pipeline complete
  Rows staged : 770,166
  Delta path  : s3://offline-store/delta/train
  Parquet out : s3://offline-store/parquet/users/staged.parquet
